# COSMoS BC+DSS Flatten — Diff vs pre-2026-03 baseline

Compares freshly regenerated `interim/COSMoS_BC_DSS.xlsx` against the snapshot under
`interim/pre-2026-03/` taken before bumping to the new COSMoS BC + DSS package.

**Scope:** flatten output only — runs *before* the QC/Compare/Parent Resolution notebooks,
so we isolate source-data changes from analysis-layer changes.

**Key:** composite `(COSMoS_BC_ID, DS_Code)` — unique across all 1545 rows in the baseline.
DS_Code is NaN for BC-only rows; normalized to empty string for the join.

**Sheets diffed:** `BC_DSS` (row-level), `Domain_Summary` (aggregate), `Statistics` (scalar).


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..') / 'interim'
BASELINE = ROOT / 'pre-2026-03'
FILE = 'COSMoS_BC_DSS.xlsx'

KEY_COLS = ['COSMoS_BC_ID', 'DS_Code']


In [ ]:
def load_bc_dss(path):
    df = pd.read_excel(path, sheet_name='BC_DSS')
    df['DS_Code'] = df['DS_Code'].fillna('')
    df['_key'] = df['COSMoS_BC_ID'].astype(str) + '||' + df['DS_Code'].astype(str)
    return df

old = load_bc_dss(BASELINE / FILE)
new = load_bc_dss(ROOT / FILE)
print(f'baseline rows: {len(old)}')
print(f'new rows:      {len(new)}')


In [ ]:
old_keys = set(old['_key'])
new_keys = set(new['_key'])
added = new[new['_key'].isin(new_keys - old_keys)].drop(columns='_key').copy()
removed = old[old['_key'].isin(old_keys - new_keys)].drop(columns='_key').copy()
print(f'added rows:   {len(added)}')
print(f'removed rows: {len(removed)}')


In [ ]:
# Cell-level changes on rows present in both
common = sorted(old_keys & new_keys)
o = old[old['_key'].isin(common)].drop(columns='_key').set_index(KEY_COLS).sort_index()
n = new[new['_key'].isin(common)].drop(columns='_key').set_index(KEY_COLS).sort_index()
cols = [c for c in o.columns if c in n.columns]
o = o[cols]; n = n[cols]

o_str = o.fillna('').astype(str)
n_str = n.fillna('').astype(str)
diff_mask = (o_str != n_str)

changed_rows = []
for idx in o.index[diff_mask.any(axis=1)]:
    for c in cols:
        if o_str.at[idx, c] != n_str.at[idx, c]:
            changed_rows.append({
                'COSMoS_BC_ID': idx[0],
                'DS_Code': idx[1],
                'column': c,
                'old': o.at[idx, c],
                'new': n.at[idx, c],
            })
changed = pd.DataFrame(changed_rows)
print(f'changed cells: {len(changed)} across {changed[["COSMoS_BC_ID","DS_Code"]].drop_duplicates().shape[0] if len(changed) else 0} rows')
if len(changed):
    print()
    print('Top changed columns:')
    print(changed["column"].value_counts())


## Inspect added rows


In [ ]:
added.head(20)


## Inspect removed rows


In [ ]:
removed.head(20)


## Inspect changed cells


In [ ]:
changed.head(40)


## Domain_Summary diff (aggregate counts)


In [ ]:
ds_old = pd.read_excel(BASELINE / FILE, sheet_name='Domain_Summary')
ds_new = pd.read_excel(ROOT / FILE, sheet_name='Domain_Summary')
merged = ds_old.merge(ds_new, on=['Domain','Domain_Label','Domain_Class'], how='outer', suffixes=('_old','_new'), indicator=True)
merged


## Statistics diff (scalar metrics)


In [ ]:
st_old = pd.read_excel(BASELINE / FILE, sheet_name='Statistics')
st_new = pd.read_excel(ROOT / FILE, sheet_name='Statistics')
st = st_old.merge(st_new, on='Metric', how='outer', suffixes=('_old','_new'))
st['changed'] = st['Value_old'].astype(str) != st['Value_new'].astype(str)
st


## Optional: write diff CSVs

Uncomment to persist diff outputs alongside the SDTM CT diffs.


In [ ]:
OUT = Path('..') / 'interim' / 'diffs' / 'pre-2026-03__current'
OUT.mkdir(parents=True, exist_ok=True)
added.to_csv(OUT / 'COSMoS_BC_DSS__added.csv', index=False)
removed.to_csv(OUT / 'COSMoS_BC_DSS__removed.csv', index=False)
changed.to_csv(OUT / 'COSMoS_BC_DSS__changed.csv', index=False)
merged.to_csv(OUT / 'COSMoS_BC_DSS__domain_summary_diff.csv', index=False)
st.to_csv(OUT / 'COSMoS_BC_DSS__statistics_diff.csv', index=False)
print('wrote', OUT)
